# S10 – Series de tiempo e inferencia en streaming (datos reales ESP32 + HC-SR04)
**Estudiante:** Lanzeloth Vargas Marichi · **Fuente:** `distanciaCm` del ESP32 físico → MQTT → bridge → Kafka `atmos-real`

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder.appName("S10-series-tiempo-real")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate())
print("Spark", spark.version)

BOOTSTRAP = "kafka:9092"
TOPIC = "atmos-real"
ART = "./artifacts"
MODEL_PATH = f"{ART}/models/modelo_distancia_lr"
OUT_PRED = f"{ART}/s10_predicciones_real"
CHK_PRED = f"{ART}/chk_s10_predicciones_real"

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-95f655fb-740a-4574-a892-ae89c3cc930d;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.mod

Spark 4.2.0


## 1. Business Understanding
**Pregunta:** dado el valor actual de `distanciaCm` (sensor ultrasónico, una lectura cada ~3 s), ¿podemos predecir la lectura del **siguiente** instante (t+1)?

- **Variable objetivo:** `siguiente = lead(distanciaCm, 1)`.
- **Feature:** solo `valor` (distancia actual). Es *sin estado*: en streaming cada evento llega solo, sin historia, y así el mismo modelo sirve en entrenamiento e inferencia.
- **Métrica:** RMSE (cm), con MAE y R² de apoyo.
- **Regla de honestidad:** cualquier modelo debe superar primero a la **persistencia** (`predicción = valor actual`). Si no la supera, no se justifica desplegarlo.

## 2. Dataset supervisado desde Kafka (datos reales) y split cronológico

In [2]:
esquema = "tipoEvento STRING, sensorId STRING, distanciaCm DOUBLE, origen STRING, timestamp LONG"

raw = (spark.read.format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load())

ev = (raw.select(F.col("timestamp").alias("kafkaTs"), F.col("offset"),
                 F.from_json(F.col("value").cast("string"), esquema).alias("e"))
        .select("kafkaTs", "offset", "e.*")
        .filter(F.col("tipoEvento").isNotNull() & F.col("distanciaCm").isNotNull() & F.col("timestamp").isNotNull())
        # rango físico válido del HC-SR04 (2–400 cm): descarta ecos fallidos
        .filter((F.col("distanciaCm") >= 2) & (F.col("distanciaCm") <= 400))
        .withColumn("tiempo", F.from_unixtime(F.col("timestamp")/1000).cast("timestamp"))
        .withColumnRenamed("distanciaCm", "valor"))
print("eventos válidos:", ev.count())
ev.orderBy("offset").show(5, truncate=False)

eventos válidos: 1764
+-----------------------+------+--------------+----------------------+-----+------------+-------------+-------------------+
|kafkaTs                |offset|tipoEvento    |sensorId              |valor|origen      |timestamp    |tiempo             |
+-----------------------+------+--------------+----------------------+-----+------------+-------------+-------------------+
|2026-10-16 02:55:58.243|9     |sensor.lectura|esp32-fisico-lanzeloth|3.3  |esp32-fisico|1792119357000|2026-10-16 02:55:57|
|2026-10-16 02:57:37.228|27    |sensor.lectura|esp32-fisico-lanzeloth|105.6|esp32-fisico|1792119456000|2026-10-16 02:57:36|
|2026-10-16 02:57:40.212|28    |sensor.lectura|esp32-fisico-lanzeloth|6.2  |esp32-fisico|1792119459000|2026-10-16 02:57:39|
|2026-10-16 02:57:43.412|29    |sensor.lectura|esp32-fisico-lanzeloth|6.9  |esp32-fisico|1792119462000|2026-10-16 02:57:42|
|2026-10-16 02:57:46.241|30    |sensor.lectura|esp32-fisico-lanzeloth|7.8  |esp32-fisico|1792119465000|2026-10

In [3]:
w = Window.orderBy("tiempo")
sup = (ev.withColumn("siguiente", F.lead("valor", 1).over(w))
         .na.drop(subset=["siguiente"])
         .select("tiempo", "valor", "siguiente").cache())
n = sup.count()
print("filas supervisadas:", n)

con_rn = sup.withColumn("rn", F.row_number().over(Window.orderBy("tiempo")))
corte = int(n * 0.8)
train = con_rn.filter(F.col("rn") <= corte).drop("rn").cache()
test  = con_rn.filter(F.col("rn") >  corte).drop("rn").cache()

r = (train.agg(F.max("tiempo").alias("fmax_train")).crossJoin(test.agg(F.min("tiempo").alias("fmin_test"))).first())
print("train:", train.count(), "| test:", test.count())
print("fecha_max_train:", r.fmax_train, "| fecha_min_test:", r.fmin_test)
assert r.fmax_train < r.fmin_test, "Fuga temporal: el split no es cronológico"
print("OK: split cronológico sin solapamiento")

26/10/22 15:14:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/22 15:14:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/22 15:14:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/22 15:14:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/22 15:14:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/22 15:14:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/22 1

filas supervisadas: 1763


26/10/22 15:14:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/22 15:14:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/22 15:14:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/22 15:14:44 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


train: 1410 | test: 353
fecha_max_train: 2026-10-16 04:08:05 | fecha_min_test: 2026-10-16 04:08:08
OK: split cronológico sin solapamiento


## 3. Baseline de persistencia (primero)

In [4]:
from pyspark.ml.evaluation import RegressionEvaluator

def metricas(df, pred="prediction"):
    out = {}
    for m in ["rmse", "mae", "r2"]:
        out[m] = RegressionEvaluator(labelCol="siguiente", predictionCol=pred, metricName=m).evaluate(df)
    return out

base = test.withColumn("prediction", F.col("valor"))
m_base = metricas(base)
print("Persistencia:", {k: round(v, 4) for k, v in m_base.items()})

Persistencia: {'rmse': 0.2016, 'mae': 0.1139, 'r2': -0.8114}


## 4. Modelado: Regresión lineal y Random Forest vs. baseline

In [5]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml import Pipeline

asm = VectorAssembler(inputCols=["valor"], outputCol="features")

pipe_lr = Pipeline(stages=[asm, LinearRegression(featuresCol="features", labelCol="siguiente")])
pipe_rf = Pipeline(stages=[asm, RandomForestRegressor(featuresCol="features", labelCol="siguiente",
                                                      numTrees=50, maxDepth=8, seed=42)])
mod_lr = pipe_lr.fit(train)
mod_rf = pipe_rf.fit(train)

m_lr = metricas(mod_lr.transform(test))
m_rf = metricas(mod_rf.transform(test))

import pandas as pd
tabla = pd.DataFrame([
    {"Modelo": "Persistencia", **m_base, "MejoraRMSE_vs_persistencia_%": 0.0},
    {"Modelo": "LinearRegression", **m_lr, "MejoraRMSE_vs_persistencia_%": 100*(m_base["rmse"]-m_lr["rmse"])/m_base["rmse"]},
    {"Modelo": "RandomForest", **m_rf, "MejoraRMSE_vs_persistencia_%": 100*(m_base["rmse"]-m_rf["rmse"])/m_base["rmse"]},
]).round(4)
tabla

26/10/22 15:16:24 WARN Instrumentation: [e4656c01] regParam is zero, which might cause numerical instability and overfitting.
netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory


,Modelo,rmse,mae,r2,MejoraRMSE_vs_persistencia_%
0,Persistencia,0.2016,0.1139,-0.8114,0.0000
1,LinearRegression,0.1821,0.1230,-0.4793,9.6302
2,RandomForest,5.2121,1.2293,-1210.3229,-2485.9683


In [8]:
for nombre, df in [("train", train), ("test", test)]:
    print(nombre)
    df.select(F.min("valor").alias("min"), F.expr("percentile_approx(valor, 0.5)").alias("mediana"),
              F.max("valor").alias("max"), F.stddev("valor").alias("std")).show()

train
+---+-------+-----+-----------------+
|min|mediana|  max|              std|
+---+-------+-----+-----------------+
|3.3|  118.6|121.7|9.544431442824498|
+---+-------+-----+-----------------+

test
+-----+-------+-----+-------------------+
|  min|mediana|  max|                std|
+-----+-------+-----+-------------------+
|118.1|  118.6|118.7|0.14996726829761997|
+-----+-------+-----+-------------------+



In [9]:
print("train con valor < 100 cm:", train.filter(F.col("valor") < 100).count(), "de", train.count())
mod_rf.transform(test).select(F.mean("prediction").alias("pred_media_rf"), F.mean("siguiente").alias("real_media")).show()

train con valor < 100 cm: 12 de 1410
+------------------+------------------+
|     pred_media_rf|        real_media|
+------------------+------------------+
|117.44918009439502|118.54589235127426|
+------------------+------------------+



## 5. Selección honesta y persistencia del modelo

In [6]:
mejora_lr = tabla.loc[1, "MejoraRMSE_vs_persistencia_%"]
mejora_rf = tabla.loc[2, "MejoraRMSE_vs_persistencia_%"]
print(f"LR: {mejora_lr:+.2f}% | RF: {mejora_rf:+.2f}% frente a persistencia")
if max(mejora_lr, mejora_rf) <= 0:
    print("Ningún modelo supera a la persistencia: se despliega LR solo como demostración técnica del pipeline.")
else:
    print("Ganador por RMSE:", "LinearRegression" if mejora_lr >= mejora_rf else "RandomForest")

# Se despliega la regresión lineal (liviana, sin estado, mismo formato de salida)
mod_lr.write().overwrite().save(MODEL_PATH)

from pyspark.ml import PipelineModel
modelo_cargado = PipelineModel.load(MODEL_PATH)
print("Etapas del modelo recargado:", [type(s).__name__ for s in modelo_cargado.stages])

# Evidencia: mismas predicciones antes y después de recargar
a = mod_lr.transform(test).select(F.col("prediction").alias("p1"), "tiempo")
b = modelo_cargado.transform(test).select(F.col("prediction").alias("p2"), "tiempo")
dif = a.join(b, "tiempo").agg(F.max(F.abs(F.col("p1")-F.col("p2")))).first()[0]
print("Diferencia máxima entre predicciones (original vs recargado):", dif)

LR: +9.63% | RF: -2485.97% frente a persistencia
Ganador por RMSE: LinearRegression


Etapas del modelo recargado: ['VectorAssembler', 'LinearRegressionModel']
Diferencia máxima entre predicciones (original vs recargado): 0.0


## 6. Inferencia en tiempo real (Kafka → modelo → Parquet)
Sin ventanas, watermark ni estado: cada evento se transforma de forma independiente dentro de `foreachBatch`.
**Requisito:** ESP32, MQTT y bridge corriendo para que lleguen eventos nuevos.

In [11]:
# Evitar dos queries sobre el mismo checkpoint
for q in spark.streams.active:
    print("Deteniendo query previa:", q.name)
    q.stop()

import shutil, os
# Si el esquema cambió respecto a una corrida anterior, limpiar checkpoint y salida
if os.path.exists(OUT_PRED):
    cols_prev = spark.read.parquet(OUT_PRED).columns
    esperadas = ["batchId", "id", "fechaHora", "valor", "prediccionSiguiente"]
    if cols_prev != esperadas:
        shutil.rmtree(OUT_PRED, ignore_errors=True); shutil.rmtree(CHK_PRED, ignore_errors=True)
        print("Esquema distinto: salida y checkpoint limpiados")

In [12]:
stream = (spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "latest")
    .option("failOnDataLoss", "false")
    .load())

eventos = (stream.select(F.col("offset").alias("id"),
                         F.from_json(F.col("value").cast("string"), esquema).alias("e"))
    .select("id", F.from_unixtime(F.col("e.timestamp")/1000).cast("timestamp").alias("fechaHora"),
            F.col("e.distanciaCm").alias("valor"))
    .filter(F.col("valor").isNotNull() & (F.col("valor") >= 2) & (F.col("valor") <= 400)))

def procesar(batch_df, batch_id):
    if batch_df.isEmpty():
        return
    pred = (modelo_cargado.transform(batch_df)
            .withColumn("batchId", F.lit(batch_id))
            .select("batchId", "id", "fechaHora", "valor", F.col("prediction").alias("prediccionSiguiente")))
    pred.write.mode("append").parquet(OUT_PRED)
    pred.show(truncate=False)

consulta = (eventos.writeStream.foreachBatch(procesar)
    .queryName("s10_inferencia_real")
    .option("checkpointLocation", CHK_PRED)
    .trigger(processingTime="5 seconds")
    .start())
print("Query activa:", consulta.isActive)

26/10/22 15:58:50 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Query activa: True


+-------+----+-------------------+-----+-------------------+
|batchId|id  |fechaHora          |valor|prediccionSiguiente|
+-------+----+-------------------+-----+-------------------+
|1      |1836|2026-10-22 15:58:53|17.9 |41.05761474807243  |
+-------+----+-------------------+-----+-------------------+



+-------+----+-------------------+-----+-------------------+
|batchId|id  |fechaHora          |valor|prediccionSiguiente|
+-------+----+-------------------+-----+-------------------+
|2      |1837|2026-10-22 15:58:56|17.9 |41.05761474807243  |
|2      |1838|2026-10-22 15:58:59|17.9 |41.05761474807243  |
+-------+----+-------------------+-----+-------------------+



+-------+----+-------------------+-----+-------------------+
|batchId|id  |fechaHora          |valor|prediccionSiguiente|
+-------+----+-------------------+-----+-------------------+
|3      |1839|2026-10-22 15:59:02|17.9 |41.05761474807243  |
+-------+----+-------------------+-----+-------------------+



...

+-------+----+-------------------+-----+-------------------+
|batchId|id  |fechaHora          |valor|prediccionSiguiente|
+-------+----+-------------------+-----+-------------------+
|64     |1940|2026-10-22 16:04:05|107.9|110.3255134160667  |
|64     |1941|2026-10-22 16:04:08|107.5|110.01765608865338 |
+-------+----+-------------------+-----+-------------------+



+-------+----+-------------------+-----+-------------------+
|batchId|id  |fechaHora          |valor|prediccionSiguiente|
+-------+----+-------------------+-----+-------------------+
|65     |1942|2026-10-22 16:04:11|107.5|110.01765608865338 |
|65     |1943|2026-10-22 16:04:14|107.5|110.01765608865338 |
+-------+----+-------------------+-----+-------------------+



+-------+----+-------------------+-----+-------------------+
|batchId|id  |fechaHora          |valor|prediccionSiguiente|
+-------+----+-------------------+-----+-------------------+
|66     |1944|2026-10-22 16:04:17|107.9|110.3255134160667  |
+-------+----+-------------------+-----+-------------------+



+-------+----+-------------------+-----+-------------------+
|batchId|id  |fechaHora          |valor|prediccionSiguiente|
+-------+----+-------------------+-----+-------------------+
|67     |1945|2026-10-22 16:04:20|107.5|110.01765608865338 |
|67     |1946|2026-10-22 16:04:23|107.8|110.24854908421335 |
+-------+----+-------------------+-----+-------------------+



+-------+----+-------------------+-----+-------------------+
|batchId|id  |fechaHora          |valor|prediccionSiguiente|
+-------+----+-------------------+-----+-------------------+
|68     |1947|2026-10-22 16:04:26|107.5|110.01765608865338 |
|68     |1948|2026-10-22 16:04:29|107.5|110.01765608865338 |
+-------+----+-------------------+-----+-------------------+



Deja correr ≥ 1–2 minutos (llegan ~20 lecturas/min) y **toma la captura** de las predicciones impresas arriba, con reloj y perfil visibles. Luego ejecuta la siguiente celda en solitario.

In [13]:
consulta.stop()
print("Query detenida")

Query detenida


26/10/22 16:04:34 WARN DAGScheduler: Failed to cancel job group a7c977df-ef89-40e2-bb70-e92e95a0eee2. Cannot find active jobs for it.
26/10/22 16:04:34 WARN DAGScheduler: Failed to cancel job group a7c977df-ef89-40e2-bb70-e92e95a0eee2. Cannot find active jobs for it.


In [14]:
res = spark.read.parquet(OUT_PRED).orderBy("id")
print("predicciones guardadas:", res.count())
res.show(15, truncate=False)

predicciones guardadas: 113
+-------+----+-------------------+-----+-------------------+
|batchId|id  |fechaHora          |valor|prediccionSiguiente|
+-------+----+-------------------+-----+-------------------+
|1      |1836|2026-10-22 15:58:53|17.9 |41.05761474807243  |
|2      |1837|2026-10-22 15:58:56|17.9 |41.05761474807243  |
|2      |1838|2026-10-22 15:58:59|17.9 |41.05761474807243  |
|3      |1839|2026-10-22 15:59:02|17.9 |41.05761474807243  |
|4      |1840|2026-10-22 15:59:05|17.9 |41.05761474807243  |
|4      |1841|2026-10-22 15:59:08|17.9 |41.05761474807243  |
|5      |1842|2026-10-22 15:59:11|17.9 |41.05761474807243  |
|5      |1843|2026-10-22 15:59:14|17.9 |41.05761474807243  |
|6      |1844|2026-10-22 15:59:17|17.9 |41.05761474807243  |
|7      |1845|2026-10-22 15:59:20|17.9 |41.05761474807243  |
|7      |1846|2026-10-22 15:59:23|18.0 |41.13457907992576  |
|8      |1847|2026-10-22 15:59:26|17.9 |41.05761474807243  |
|8      |1848|2026-10-22 15:59:29|17.9 |41.05761474807243

In [15]:
from pyspark.sql.window import Window
live = (spark.read.parquet(OUT_PRED).orderBy("id")
        .withColumn("real_siguiente", F.lead("valor", 1).over(Window.orderBy("id")))
        .na.drop(subset=["real_siguiente"]))
for nombre, col in [("Modelo LR (desplegado)", "prediccionSiguiente"), ("Persistencia", "valor")]:
    m = live.select(F.sqrt(F.mean((F.col(col) - F.col("real_siguiente"))**2)).alias("rmse"),
                    F.mean(F.abs(F.col(col) - F.col("real_siguiente"))).alias("mae")).first()
    print(f"{nombre}: RMSE={m.rmse:.3f} | MAE={m.mae:.3f}")

26/10/22 16:06:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/22 16:06:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/22 16:06:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/22 16:06:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/22 16:06:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/22 16:06:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/22 1

Modelo LR (desplegado): RMSE=21.879 | MAE=19.968
Persistencia: RMSE=8.618 | MAE=1.254


26/10/22 16:06:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/22 16:06:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/22 16:06:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/22 16:06:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                

## 7. Gráfico (opcional)

In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt
tr = train.orderBy(F.col("tiempo").desc()).limit(60).orderBy("tiempo").toPandas()
te = mod_lr.transform(test).orderBy(F.col("tiempo").desc()).limit(60).orderBy("tiempo").toPandas()
st = res.orderBy("fechaHora").toPandas()

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(tr["tiempo"], tr["valor"], label="train (cola)", color="tab:blue")
ax.plot(te["tiempo"], te["valor"], label="test (cola)", color="tab:orange")
ax.plot(st["fechaHora"], st["valor"], label="streaming real", color="tab:green")
ax.scatter(st["fechaHora"], st["prediccionSiguiente"], label="predicción t+1", color="tab:red", s=18, zorder=3)
ax.set_ylabel("distanciaCm"); ax.set_title("Serie real de distancia y predicción t+1"); ax.legend()
plt.tight_layout(); plt.show()